# Enhanced Multi-File DEXA Data Processor

## Advanced batch processing system for comprehensive DEXA analysis

**Multi-File Processing**: Handle 5+ files simultaneously with batch operations
**Flexible File Types**: Support for .txt, .csv, .xlsx, .pdf, images (.tif, .png, .jpeg, .bmp)
**Intelligent Data Cleaning**: Advanced test detection and duplicate removal
**Smart Imputation**: Multiple strategies for missing data with group-based intelligence
**Unified Format**: Standardize timepoints and batch naming across all data sources
**Export Options**: Generate clean CSV/Excel with comprehensive summaries

*Professional toolkit for research-grade DEXA data processing and analysis*

## 1. Import Required Libraries and Setup

Setting up the environment with all necessary libraries for comprehensive DEXA data processing.

In [ ]:
# Essential data processing libraries
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# File processing libraries
from PIL import Image
import zipfile
import tempfile
import uuid

# Optional libraries for advanced file support
try:
    import pdfplumber
    PDF_SUPPORT = True
    print("PDF support enabled")
except ImportError:
    PDF_SUPPORT = False
    print("Warning: PDF support not available - install pdfplumber for PDF processing")

try:
    import xlrd
    import openpyxl
    EXCEL_SUPPORT = True
    print("Advanced Excel support enabled")
except ImportError:
    EXCEL_SUPPORT = False
    print("Warning: Limited Excel support - install xlrd and openpyxl for full .xls/.xlsx processing")

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')

# Configuration variables
MAX_FILES = 50  # Maximum files to process in one batch
MAX_FILE_SIZE_MB = 50  # Maximum file size in MB
SUPPORTED_EXTENSIONS = ['.txt', '.csv', '.xlsx', '.xls', '.pdf', '.tif', '.tiff', '.png', '.jpeg', '.jpg', '.img', '.pxl', '.bmp']

print("Enhanced DEXA Processor Initialized")
print(f"Supported file types: {', '.join(SUPPORTED_EXTENSIONS)}")
print(f"Max files per batch: {MAX_FILES}")
print(f"Max file size: {MAX_FILE_SIZE_MB}MB")

## 2. Multi-File Processing Functions

Advanced functions to handle batch processing of multiple DEXA files with validation, progress tracking, and error handling.

In [ ]:
class BatchFileProcessor:
    """
    Advanced batch file processor for DEXA data with intelligent file handling,
    validation, and progress tracking for processing 5+ files simultaneously.
    """
    
    def __init__(self):
        self.processed_files = []
        self.failed_files = []
        self.errors = []
        self.stats = {}
        
    def validate_files(self, file_paths):
        """Validate file paths and extensions before processing"""
        valid_files = []
        invalid_files = []
        
        for file_path in file_paths:
            if not os.path.exists(file_path):
                invalid_files.append(f"File not found: {file_path}")
                continue
                
            # Check file extension
            file_ext = Path(file_path).suffix.lower()
            if file_ext not in SUPPORTED_EXTENSIONS:
                invalid_files.append(f"Unsupported format: {file_path} ({file_ext})")
                continue
                
            # Check file size
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            if file_size_mb > MAX_FILE_SIZE_MB:
                invalid_files.append(f"File too large: {file_path} ({file_size_mb:.1f}MB)")
                continue
                
            valid_files.append(file_path)
            
        if invalid_files:
            print(f"Validation issues found:")
            for issue in invalid_files:
                print(f"  - {issue}")
                
        return valid_files, invalid_files
    
    def load_all_batch_data(self, file_paths, imputation_strategy='smart'):
        """
        Load and process multiple DEXA files with comprehensive error handling
        
        Parameters:
        -----------
        file_paths : list
            List of file paths to process
        imputation_strategy : str
            Strategy for handling missing data: 'leave_nan', 'zero', 'group_median', 'forward_fill', 'smart'
        
        Returns:
        --------
        pandas.DataFrame : Combined and cleaned dataset
        """
        # Validate inputs
        if not file_paths:
            raise ValueError("No file paths provided")
            
        if len(file_paths) > MAX_FILES:
            print(f"Warning: Too many files ({len(file_paths)}). Processing first {MAX_FILES}")
            file_paths = file_paths[:MAX_FILES]
        
        # Validate files
        files_to_process, invalid_files = self.validate_files(file_paths)
        
        if not files_to_process:
            raise ValueError("No valid files to process")
        
        print(f"Starting batch processing of {len(files_to_process)} files...")
        
        all_data = []
        successful_files = 0
        
        for i, file_path in enumerate(files_to_process):
            try:
                self.print_progress(i + 1, len(files_to_process), os.path.basename(file_path))
                
                # Load file based on extension
                file_ext = Path(file_path).suffix.lower()
                
                if file_ext in ['.txt', '.csv']:
                    df = self.parse_txt_csv_file(file_path)
                elif file_ext in ['.xlsx', '.xls']:
                    df = self.parse_excel_file(file_path)
                elif file_ext == '.pdf':
                    df = self.parse_pdf_file(file_path)
                elif file_ext in ['.tif', '.tiff', '.png', '.jpeg', '.jpg', '.bmp']:
                    df = self.scan_dexa_images([file_path])
                else:
                    raise ValueError(f"Unsupported file type: {file_ext}")
                
                if df is not None and not df.empty:
                    # Add metadata
                    df['source_file'] = os.path.basename(file_path)
                    df['file_type'] = file_ext
                    df['processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    
                    all_data.append(df)
                    successful_files += 1
                    self.processed_files.append(file_path)
                else:
                    self.failed_files.append(file_path)
                    self.errors.append(f"No data extracted from {file_path}")
                    
            except Exception as e:
                error_msg = f"Error processing {file_path}: {str(e)}"
                self.errors.append(error_msg)
                self.failed_files.append(file_path)
                print(f"  Error: {error_msg}")
        
        if not all_data:
            raise ValueError("No data could be extracted from any files")
        
        # Combine all data
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Clean and standardize the combined dataset
        cleaned_df = self.comprehensive_data_cleaning(combined_df)
        
        # Apply imputation strategy
        final_df = self.smart_missing_data_imputation(cleaned_df, strategy=imputation_strategy)
        
        # Store processing statistics
        self.stats = {
            'total_files_attempted': len(files_to_process),
            'successful_files': successful_files,
            'failed_files': len(self.failed_files),
            'total_records': len(final_df),
            'data_quality_score': self.calculate_data_quality_score(final_df)
        }
        
        print(f"Batch processing complete!")
        print(f"Processed: {successful_files}/{len(files_to_process)} files")
        print(f"Total records: {len(all_data)}")
        
        if self.errors:
            print(f"Errors encountered: {len(self.errors)}")
            for error in self.errors[:3]:  # Show first 3 errors
                print(f"  - {error}")
        
        return final_df
    
    def print_progress(self, current, total, filename):
        """Print processing progress"""
        progress = (current / total) * 100
        print(f"Processing {current}/{total} ({progress:.1f}%): {filename}")

print("BatchFileProcessor class created - ready for multi-file operations!")

## 3. Flexible File Type Detection and Parsing

Enhanced file parsing with automatic type detection and format-specific processing for all supported DEXA file types.

In [ ]:
def comprehensive_data_cleaning(df):
    """
    Enhanced data cleaning with comprehensive validation, outlier detection,
    and quality assessment for multi-batch DEXA data processing.
    """
    print("Starting comprehensive data cleaning...")
    
    # Initialize cleaning report
    cleaning_report = {
        'original_records': len(df),
        'test_records_removed': 0,
        'invalid_records_removed': 0,
        'outliers_removed': 0,
        'duplicates_removed': 0,
        'issues_found': []
    }
    
    if df.empty:
        print("Warning: Empty dataset provided")
        return df, cleaning_report
    
    cleaned_df = df.copy()
    initial_count = len(cleaned_df)
    
    # 1. ENHANCED TEST DATA DETECTION
    print("  Detecting and removing test data...")
    
    test_patterns = [
        r'^test', r'^demo', r'^example', r'^sample', r'^dummy',
        r'test$', r'_test', r'-test', r'\.test',
        r'^calib', r'^cal', r'calibration', r'phantom',
        r'^qc', r'quality.*control', r'^qa',
        r'training', r'practice', r'trial',
        r'unknown', r'tbd', r'placeholder'
    ]
    
    test_columns = ['subject_id', 'batch', 'timepoint', 'gender', 'filename']
    
    for col in test_columns:
        if col in cleaned_df.columns:
            for pattern in test_patterns:
                mask = cleaned_df[col].astype(str).str.contains(pattern, case=False, na=False)
                test_rows = mask.sum()
                if test_rows > 0:
                    print(f"    Removing {test_rows} {pattern} rows from {col}")
                    cleaned_df = cleaned_df[~mask]
    
    cleaning_report['test_records_removed'] = initial_count - len(cleaned_df)
    
    # 2. CRITICAL FIELD VALIDATION
    print("  Validating critical fields...")
    
    critical_fields = ['subject_id', 'batch', 'timepoint']
    before_validation = len(cleaned_df)
    
    for field in critical_fields:
        if field in cleaned_df.columns:
            # Remove null/empty values
            null_mask = cleaned_df[field].isnull() | (cleaned_df[field].astype(str).str.strip() == '')
            null_count = null_mask.sum()
            if null_count > 0:
                print(f"    Removing {null_count} records with missing {field}")
                cleaned_df = cleaned_df[~null_mask]
                cleaning_report['issues_found'].append(f"Missing {field}: {null_count} records")
    
    cleaning_report['invalid_records_removed'] = before_validation - len(cleaned_df)
    
    # 3. ENHANCED MEASUREMENT VALIDATION
    print("  Validating DEXA measurements...")
    
    measurement_fields = ['total_weight', 'soft_weight', 'lean_weight', 'fat_weight', 
                         'fat_percent', 'bmc', 'bmd', 'bone_area', 'sample_area']
    
    validation_rules = {
        'total_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'soft_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'lean_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'fat_weight': {'min': 0, 'max': 100, 'unit': 'g'},
        'fat_percent': {'min': 0, 'max': 100, 'unit': '%'},
        'bmc': {'min': 0, 'max': 10, 'unit': 'g'},
        'bmd': {'min': 0, 'max': 200, 'unit': 'mg/cm²'},
        'bone_area': {'min': 0, 'max': 50, 'unit': 'cm²'},
        'sample_area': {'min': 0, 'max': 100, 'unit': 'cm²'}
    }
    
    outliers_before = len(cleaned_df)
    
    for field in measurement_fields:
        if field not in cleaned_df.columns:
            continue
            
        # Convert to numeric first
        cleaned_df[field] = pd.to_numeric(cleaned_df[field], errors='coerce')
        
        rules = validation_rules.get(field, {})
        
        # Apply range validation
        if 'min' in rules and 'max' in rules:
            valid_range = (cleaned_df[field] >= rules['min']) & (cleaned_df[field] <= rules['max'])
            invalid_count = (~valid_range & cleaned_df[field].notna()).sum()
            
            if invalid_count > 0:
                print(f"    Removing {invalid_count} records with invalid {field} values")
                cleaned_df = cleaned_df[valid_range | cleaned_df[field].isna()]
                cleaning_report['issues_found'].append(f"Invalid {field} range: {invalid_count} records")
        
        # Statistical outlier detection (more sophisticated)
        if len(cleaned_df) > 10 and cleaned_df[field].notna().sum() > 5:
            field_data = cleaned_df[field].dropna()
            
            if len(field_data) > 0 and field_data.std() > 0:
                # Use IQR method for outlier detection
                Q1 = field_data.quantile(0.25)
                Q3 = field_data.quantile(0.75)
                IQR = Q3 - Q1
                
                # More lenient outlier bounds
                lower_bound = Q1 - 2.5 * IQR
                upper_bound = Q3 + 2.5 * IQR
                
                outlier_mask = (cleaned_df[field] < lower_bound) | (cleaned_df[field] > upper_bound)
                outlier_count = (outlier_mask & cleaned_df[field].notna()).sum()
                
                if outlier_count > 0 and outlier_count < len(cleaned_df) * 0.1:  # Don't remove more than 10%
                    print(f"    Removing {outlier_count} statistical outliers in {field}")
                    cleaned_df = cleaned_df[~outlier_mask | cleaned_df[field].isna()]
                    cleaning_report['issues_found'].append(f"Statistical outliers in {field}: {outlier_count} records")
    
    cleaning_report['outliers_removed'] = outliers_before - len(cleaned_df)
    
    # 4. ENHANCED BATCH AND TIMEPOINT STANDARDIZATION
    print("  Standardizing batch and timepoint naming...")
    
    # Enhanced batch mapping with more patterns
    if 'batch' in cleaned_df.columns:
        batch_mapping = {
            # Standard patterns
            'batch1': 'Batch_1', 'batch_1': 'Batch_1', 'b1': 'Batch_1', 'batch 1': 'Batch_1',
            'batch2': 'Batch_2', 'batch_2': 'Batch_2', 'b2': 'Batch_2', 'batch 2': 'Batch_2',
            'batch3': 'Batch_3', 'batch_3': 'Batch_3', 'b3': 'Batch_3', 'batch 3': 'Batch_3',
            'batch4': 'Batch_4', 'batch_4': 'Batch_4', 'b4': 'Batch_4', 'batch 4': 'Batch_4',
            'batch5': 'Batch_5', 'batch_5': 'Batch_5', 'b5': 'Batch_5', 'batch 5': 'Batch_5',
            # Extended patterns
            'batch6': 'Batch_6', 'batch_6': 'Batch_6', 'b6': 'Batch_6', 'batch 6': 'Batch_6',
            'batch7': 'Batch_7', 'batch_7': 'Batch_7', 'b7': 'Batch_7', 'batch 7': 'Batch_7',
            'batch8': 'Batch_8', 'batch_8': 'Batch_8', 'b8': 'Batch_8', 'batch 8': 'Batch_8',
            'batch9': 'Batch_9', 'batch_9': 'Batch_9', 'b9': 'Batch_9', 'batch 9': 'Batch_9',
            'batch10': 'Batch_10', 'batch_10': 'Batch_10', 'b10': 'Batch_10', 'batch 10': 'Batch_10',
        }
        
        # Clean and standardize batch names
        cleaned_df['batch'] = cleaned_df['batch'].astype(str).str.strip().str.lower()
        cleaned_df['batch'] = cleaned_df['batch'].map(batch_mapping).fillna(cleaned_df['batch'])
        
        # Count standardized batches
        unique_batches = cleaned_df['batch'].nunique()
        print(f"    Standardized {unique_batches} unique batches")
    
    # Enhanced timepoint standardization
    if 'timepoint' in cleaned_df.columns:
        timepoint_mapping = {
            # Baseline variations
            'week_0': 'Baseline', 'week0': 'Baseline', 'week 0': 'Baseline',
            'pre_scan': 'Baseline', 'prescan': 'Baseline', 'pre scan': 'Baseline',
            'baseline': 'Baseline', 'pre': 'Baseline', 'week-1': 'Baseline',
            # Week patterns
            'week_1': 'Week_1', 'week1': 'Week_1', 'week 1': 'Week_1',
            'week_2': 'Week_2', 'week2': 'Week_2', 'week 2': 'Week_2',
            'week_3': 'Week_3', 'week3': 'Week_3', 'week 3': 'Week_3',
            'week_4': 'Week_4', 'week4': 'Week_4', 'week 4': 'Week_4',
            'week_5': 'Week_5', 'week5': 'Week_5', 'week 5': 'Week_5',
            # Post patterns
            'post_scan': 'Post_Scan', 'postscan': 'Post_Scan', 'post scan': 'Post_Scan',
            'post_treatment': 'Post_Scan', 'post treatment': 'Post_Scan',
            'final': 'Post_Scan', 'end': 'Post_Scan',
            # Other
            'root': 'Unknown', 'unknown': 'Unknown'
        }
        
        # Clean and standardize timepoint names
        cleaned_df['timepoint'] = cleaned_df['timepoint'].astype(str).str.strip().str.lower()
        cleaned_df['timepoint_original'] = cleaned_df['timepoint'].copy()
        cleaned_df['timepoint_standardized'] = cleaned_df['timepoint'].map(timepoint_mapping).fillna(cleaned_df['timepoint'])
        
        unique_timepoints = cleaned_df['timepoint_standardized'].nunique()
        print(f"    Standardized {unique_timepoints} unique timepoints")
    
    # 5. SUBJECT ID CLEANING
    print("  Cleaning subject IDs...")
    
    if 'subject_id' in cleaned_df.columns:
        # Remove problematic subject IDs
        problematic_ids = ['', 'nan', 'none', 'null', 'unknown', 'missing']
        before_id_clean = len(cleaned_df)
        
        cleaned_df['subject_id'] = cleaned_df['subject_id'].astype(str).str.strip()
        id_mask = ~cleaned_df['subject_id'].str.lower().isin(problematic_ids)
        cleaned_df = cleaned_df[id_mask]
        
        id_clean_count = before_id_clean - len(cleaned_df)
        if id_clean_count > 0:
            print(f"    Removed {id_clean_count} records with invalid subject IDs")
    
    # 6. DUPLICATE DETECTION AND REMOVAL
    print("  Detecting and removing duplicates...")
    
    duplicates_before = len(cleaned_df)
    
    # Define columns for duplicate detection (exclude filename for logical duplicates)
    duplicate_columns = ['batch', 'subject_id', 'timepoint', 'gender'] + \
                       [col for col in measurement_fields if col in cleaned_df.columns]
    
    # Remove exact duplicates
    cleaned_df = cleaned_df.drop_duplicates(subset=duplicate_columns, keep='first')
    
    duplicates_removed = duplicates_before - len(cleaned_df)
    cleaning_report['duplicates_removed'] = duplicates_removed
    
    if duplicates_removed > 0:
        print(f"    Removed {duplicates_removed} duplicate records")
    
    # 7. DATA QUALITY ASSESSMENT
    print("  Assessing data quality...")
    
    quality_metrics = {
        'completeness': 0,
        'validity': 0,
        'consistency': 0,
        'accuracy': 0
    }
    
    # Completeness: percentage of non-null values in key fields
    key_fields = ['batch', 'subject_id', 'timepoint', 'gender'] + measurement_fields[:4]
    total_possible_values = len(cleaned_df) * len(key_fields)
    non_null_values = sum(cleaned_df[field].notna().sum() for field in key_fields if field in cleaned_df.columns)
    quality_metrics['completeness'] = (non_null_values / total_possible_values) * 100 if total_possible_values > 0 else 0
    
    # Validity: percentage of values within expected ranges
    valid_count = len(cleaned_df)  # All remaining records passed validation
    total_count = cleaning_report['original_records']
    quality_metrics['validity'] = (valid_count / total_count) * 100 if total_count > 0 else 0
    
    # Consistency: standardization success rate
    if 'batch' in cleaned_df.columns and 'timepoint_standardized' in cleaned_df.columns:
        standardized_batches = cleaned_df['batch'].str.startswith('Batch_').sum()
        standardized_timepoints = cleaned_df['timepoint_standardized'].isin(['Baseline', 'Week_1', 'Week_2', 'Week_3', 'Week_4', 'Week_5', 'Post_Scan']).sum()
        quality_metrics['consistency'] = ((standardized_batches + standardized_timepoints) / (2 * len(cleaned_df))) * 100
    
    # Accuracy: inverse of outliers removed (higher is better)
    outlier_rate = cleaning_report['outliers_removed'] / cleaning_report['original_records'] if cleaning_report['original_records'] > 0 else 0
    quality_metrics['accuracy'] = max(0, (1 - outlier_rate) * 100)
    
    # Overall quality score (weighted average)
    overall_quality = (
        quality_metrics['completeness'] * 0.3 +
        quality_metrics['validity'] * 0.3 +
        quality_metrics['consistency'] * 0.2 +
        quality_metrics['accuracy'] * 0.2
    )
    
    cleaning_report['data_quality_score'] = round(overall_quality, 2)
    cleaning_report['quality_metrics'] = quality_metrics
    cleaning_report['final_records'] = len(cleaned_df)
    
    # FINAL REPORT
    print("\nCOMPREHENSIVE CLEANING REPORT:")
    print(f"  Original records: {cleaning_report['original_records']}")
    print(f"  Test records removed: {cleaning_report['test_records_removed']}")
    print(f"  Invalid records removed: {cleaning_report['invalid_records_removed']}")
    print(f"  Outliers removed: {cleaning_report['outliers_removed']}")
    print(f"  Duplicates removed: {cleaning_report['duplicates_removed']}")
    print(f"  Final records: {cleaning_report['final_records']}")
    print(f"  Data quality score: {cleaning_report['data_quality_score']:.1f}%")
    
    if cleaning_report['issues_found']:
        print(f"  Issues addressed: {len(cleaning_report['issues_found'])}")
        for issue in cleaning_report['issues_found'][:5]:  # Show first 5
            print(f"    • {issue}")
        if len(cleaning_report['issues_found']) > 5:
            print(f"    ... and {len(cleaning_report['issues_found']) - 5} more")
    
    return cleaned_df, cleaning_report

print("Comprehensive data cleaning functions created!")
print("Features: Advanced test detection, statistical validation, quality scoring")

## 4. Advanced Data Cleaning and Validation

Comprehensive cleaning functions to remove test data, validate measurements, and standardize formats across all batches.

In [ ]:
def clean_dexa_data_comprehensive(df):
    """
    Comprehensive DEXA data cleaning with enhanced validation and reporting.
    
    Features:
    - Advanced test data detection and removal
    - Intelligent outlier detection with statistical validation
    - Multi-level data validation (syntactic, semantic, statistical)
    - Batch and timepoint standardization
    - Data quality scoring and reporting
    """
    
    print("Starting comprehensive DEXA data cleaning...")
    
    cleaning_report = {
        'original_records': len(df),
        'test_records_removed': 0,
        'invalid_records_removed': 0,
        'outliers_removed': 0,
        'duplicates_removed': 0,
        'final_records': 0,
        'data_quality_score': 0,
        'issues_found': []
    }
    
    cleaned_df = df.copy()
    
    # 1. ENHANCED TEST DATA REMOVAL
    print("  Detecting and removing test/calibration data...")
    
    test_patterns = [
        'test', 'TEST', 'Test', 'calibration', 'CALIBRATION', 'blank', 'BLANK',
        'standard', 'STANDARD', 'control', 'CONTROL', 'phantom', 'PHANTOM',
        'demo', 'DEMO', 'sample', 'SAMPLE', 'qc', 'QC', 'quality', 'QUALITY',
        'check', 'CHECK', 'verify', 'VERIFY', 'calib', 'CALIB'
    ]
    
    initial_count = len(cleaned_df)
    test_columns = ['subject_id', 'filename']
    
    for col in test_columns:
        if col in cleaned_df.columns:
            for pattern in test_patterns:
                mask = cleaned_df[col].astype(str).str.contains(pattern, case=False, na=False)
                test_rows = mask.sum()
                if test_rows > 0:
                    print(f"    Removing {test_rows} {pattern} rows from {col}")
                    cleaned_df = cleaned_df[~mask]
    
    cleaning_report['test_records_removed'] = initial_count - len(cleaned_df)
    
    # 2. CRITICAL FIELD VALIDATION
    print("  Validating critical fields...")
    
    critical_fields = ['subject_id', 'batch', 'timepoint']
    before_validation = len(cleaned_df)
    
    for field in critical_fields:
        if field in cleaned_df.columns:
            # Remove null/empty values
            null_mask = cleaned_df[field].isnull() | (cleaned_df[field].astype(str).str.strip() == '')
            null_count = null_mask.sum()
            if null_count > 0:
                print(f"    Removing {null_count} records with missing {field}")
                cleaned_df = cleaned_df[~null_mask]
                cleaning_report['issues_found'].append(f"Missing {field}: {null_count} records")
    
    cleaning_report['invalid_records_removed'] = before_validation - len(cleaned_df)
    
    # 3. ENHANCED MEASUREMENT VALIDATION
    print("  Validating DEXA measurements...")
    
    measurement_fields = ['total_weight', 'soft_weight', 'lean_weight', 'fat_weight', 
                         'fat_percent', 'bmc', 'bmd', 'bone_area', 'sample_area']
    
    validation_rules = {
        'total_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'soft_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'lean_weight': {'min': 0, 'max': 200, 'unit': 'g'},
        'fat_weight': {'min': 0, 'max': 100, 'unit': 'g'},
        'fat_percent': {'min': 0, 'max': 100, 'unit': '%'},
        'bmc': {'min': 0, 'max': 10, 'unit': 'g'},
        'bmd': {'min': 0, 'max': 200, 'unit': 'mg/cm²'},
        'bone_area': {'min': 0, 'max': 50, 'unit': 'cm²'},
        'sample_area': {'min': 0, 'max': 100, 'unit': 'cm²'}
    }
    
    outliers_before = len(cleaned_df)
    
    for field in measurement_fields:
        if field not in cleaned_df.columns:
            continue
            
        # Convert to numeric first
        cleaned_df[field] = pd.to_numeric(cleaned_df[field], errors='coerce')
        
        rules = validation_rules.get(field, {})
        
        # Apply range validation
        if 'min' in rules and 'max' in rules:
            valid_range = (cleaned_df[field] >= rules['min']) & (cleaned_df[field] <= rules['max'])
            invalid_count = (~valid_range & cleaned_df[field].notna()).sum()
            
            if invalid_count > 0:
                print(f"    Removing {invalid_count} records with invalid {field} values")
                cleaned_df = cleaned_df[valid_range | cleaned_df[field].isna()]
                cleaning_report['issues_found'].append(f"Invalid {field} range: {invalid_count} records")
        
        # Statistical outlier detection (more sophisticated)
        if len(cleaned_df) > 10 and cleaned_df[field].notna().sum() > 5:
            field_data = cleaned_df[field].dropna()
            
            if len(field_data) > 0 and field_data.std() > 0:
                # Use IQR method for outlier detection
                Q1 = field_data.quantile(0.25)
                Q3 = field_data.quantile(0.75)
                IQR = Q3 - Q1
                
                # More lenient outlier bounds
                lower_bound = Q1 - 2.5 * IQR
                upper_bound = Q3 + 2.5 * IQR
                
                outlier_mask = (cleaned_df[field] < lower_bound) | (cleaned_df[field] > upper_bound)
                outlier_count = (outlier_mask & cleaned_df[field].notna()).sum()
                
                if outlier_count > 0 and outlier_count < len(cleaned_df) * 0.1:  # Don't remove more than 10%
                    print(f"    Removing {outlier_count} statistical outliers in {field}")
                    cleaned_df = cleaned_df[~outlier_mask | cleaned_df[field].isna()]
                    cleaning_report['issues_found'].append(f"Statistical outliers in {field}: {outlier_count} records")
    
    cleaning_report['outliers_removed'] = outliers_before - len(cleaned_df)
    
    # 4. ENHANCED BATCH AND TIMEPOINT STANDARDIZATION
    print("  Standardizing batch and timepoint naming...")
    
    # Enhanced batch mapping with more patterns
    if 'batch' in cleaned_df.columns:
        batch_mapping = {
            # Standard patterns
            'batch1': 'Batch_1', 'batch_1': 'Batch_1', 'b1': 'Batch_1', 'batch 1': 'Batch_1',
            'batch2': 'Batch_2', 'batch_2': 'Batch_2', 'b2': 'Batch_2', 'batch 2': 'Batch_2',
            'batch3': 'Batch_3', 'batch_3': 'Batch_3', 'b3': 'Batch_3', 'batch 3': 'Batch_3',
            'batch4': 'Batch_4', 'batch_4': 'Batch_4', 'b4': 'Batch_4', 'batch 4': 'Batch_4',
            'batch5': 'Batch_5', 'batch_5': 'Batch_5', 'b5': 'Batch_5', 'batch 5': 'Batch_5',
            # Extended patterns
            'batch6': 'Batch_6', 'batch_6': 'Batch_6', 'b6': 'Batch_6', 'batch 6': 'Batch_6',
            'batch7': 'Batch_7', 'batch_7': 'Batch_7', 'b7': 'Batch_7', 'batch 7': 'Batch_7',
            'batch8': 'Batch_8', 'batch_8': 'Batch_8', 'b8': 'Batch_8', 'batch 8': 'Batch_8',
            'batch9': 'Batch_9', 'batch_9': 'Batch_9', 'b9': 'Batch_9', 'batch 9': 'Batch_9',
            'batch10': 'Batch_10', 'batch_10': 'Batch_10', 'b10': 'Batch_10', 'batch 10': 'Batch_10',
        }
        
        # Clean and standardize batch names
        cleaned_df['batch'] = cleaned_df['batch'].astype(str).str.strip().str.lower()
        cleaned_df['batch'] = cleaned_df['batch'].map(batch_mapping).fillna(cleaned_df['batch'])
        
        # Count standardized batches
        unique_batches = cleaned_df['batch'].nunique()
        print(f"    Standardized {unique_batches} unique batches")
    
    # Enhanced timepoint standardization
    if 'timepoint' in cleaned_df.columns:
        timepoint_mapping = {
            # Baseline variations
            'week_0': 'Baseline', 'week0': 'Baseline', 'week 0': 'Baseline',
            'pre_scan': 'Baseline', 'prescan': 'Baseline', 'pre scan': 'Baseline',
            'baseline': 'Baseline', 'pre': 'Baseline', 'week-1': 'Baseline',
            # Week patterns
            'week_1': 'Week_1', 'week1': 'Week_1', 'week 1': 'Week_1',
            'week_2': 'Week_2', 'week2': 'Week_2', 'week 2': 'Week_2',
            'week_3': 'Week_3', 'week3': 'Week_3', 'week 3': 'Week_3',
            'week_4': 'Week_4', 'week4': 'Week_4', 'week 4': 'Week_4',
            'week_5': 'Week_5', 'week5': 'Week_5', 'week 5': 'Week_5',
            # Post patterns
            'post_scan': 'Post_Scan', 'postscan': 'Post_Scan', 'post scan': 'Post_Scan',
            'post_treatment': 'Post_Scan', 'post treatment': 'Post_Scan',
            'final': 'Post_Scan', 'end': 'Post_Scan',
            # Other
            'root': 'Unknown', 'unknown': 'Unknown'
        }
        
        # Clean and standardize timepoint names
        cleaned_df['timepoint'] = cleaned_df['timepoint'].astype(str).str.strip().str.lower()
        cleaned_df['timepoint_original'] = cleaned_df['timepoint'].copy()
        cleaned_df['timepoint_standardized'] = cleaned_df['timepoint'].map(timepoint_mapping).fillna(cleaned_df['timepoint'])
        
        unique_timepoints = cleaned_df['timepoint_standardized'].nunique()
        print(f"    Standardized {unique_timepoints} unique timepoints")
    
    # 5. SUBJECT ID CLEANING
    print("  Cleaning subject IDs...")
    
    if 'subject_id' in cleaned_df.columns:
        # Remove problematic subject IDs
        problematic_ids = ['', 'nan', 'none', 'null', 'unknown', 'missing']
        before_id_clean = len(cleaned_df)
        
        cleaned_df['subject_id'] = cleaned_df['subject_id'].astype(str).str.strip()
        id_mask = ~cleaned_df['subject_id'].str.lower().isin(problematic_ids)
        cleaned_df = cleaned_df[id_mask]
        
        id_clean_count = before_id_clean - len(cleaned_df)
        if id_clean_count > 0:
            print(f"    Removed {id_clean_count} records with invalid subject IDs")
    
    # 6. DUPLICATE DETECTION AND REMOVAL
    print("  Detecting and removing duplicates...")
    
    duplicates_before = len(cleaned_df)
    
    # Define columns for duplicate detection (exclude filename for logical duplicates)
    duplicate_columns = ['batch', 'subject_id', 'timepoint', 'gender'] + \
                       [col for col in measurement_fields if col in cleaned_df.columns]
    
    # Remove exact duplicates
    cleaned_df = cleaned_df.drop_duplicates(subset=duplicate_columns, keep='first')
    
    duplicates_removed = duplicates_before - len(cleaned_df)
    cleaning_report['duplicates_removed'] = duplicates_removed
    
    if duplicates_removed > 0:
        print(f"    Removed {duplicates_removed} duplicate records")
    
    # 7. DATA QUALITY ASSESSMENT
    print("  Assessing data quality...")
    
    quality_metrics = {
        'completeness': 0,
        'validity': 0,
        'consistency': 0,
        'accuracy': 0
    }
    
    # Completeness: percentage of non-null values in key fields
    key_fields = ['batch', 'subject_id', 'timepoint', 'gender'] + measurement_fields[:4]
    total_possible_values = len(cleaned_df) * len(key_fields)
    non_null_values = sum(cleaned_df[field].notna().sum() for field in key_fields if field in cleaned_df.columns)
    quality_metrics['completeness'] = (non_null_values / total_possible_values) * 100 if total_possible_values > 0 else 0
    
    # Validity: percentage of values within expected ranges
    valid_count = len(cleaned_df)  # All remaining records passed validation
    total_count = cleaning_report['original_records']
    quality_metrics['validity'] = (valid_count / total_count) * 100 if total_count > 0 else 0
    
    # Consistency: standardization success rate
    if 'batch' in cleaned_df.columns and 'timepoint_standardized' in cleaned_df.columns:
        standardized_batches = cleaned_df['batch'].str.startswith('Batch_').sum()
        standardized_timepoints = cleaned_df['timepoint_standardized'].isin(['Baseline', 'Week_1', 'Week_2', 'Week_3', 'Week_4', 'Week_5', 'Post_Scan']).sum()
        quality_metrics['consistency'] = ((standardized_batches + standardized_timepoints) / (2 * len(cleaned_df))) * 100
    
    # Accuracy: inverse of outliers removed (higher is better)
    outlier_rate = cleaning_report['outliers_removed'] / cleaning_report['original_records'] if cleaning_report['original_records'] > 0 else 0
    quality_metrics['accuracy'] = max(0, (1 - outlier_rate) * 100)
    
    # Overall quality score (weighted average)
    overall_quality = (
        quality_metrics['completeness'] * 0.3 +
        quality_metrics['validity'] * 0.3 +
        quality_metrics['consistency'] * 0.2 +
        quality_metrics['accuracy'] * 0.2
    )
    
    cleaning_report['data_quality_score'] = round(overall_quality, 2)
    cleaning_report['quality_metrics'] = quality_metrics
    cleaning_report['final_records'] = len(cleaned_df)
    
    # FINAL REPORT
    print("\nCOMPREHENSIVE CLEANING REPORT:")
    print(f"  Original records: {cleaning_report['original_records']}")
    print(f"  Test records removed: {cleaning_report['test_records_removed']}")
    print(f"  Invalid records removed: {cleaning_report['invalid_records_removed']}")
    print(f"  Outliers removed: {cleaning_report['outliers_removed']}")
    print(f"  Duplicates removed: {cleaning_report['duplicates_removed']}")
    print(f"  Final records: {cleaning_report['final_records']}")
    print(f"  Data quality score: {cleaning_report['data_quality_score']:.1f}%")
    
    if cleaning_report['issues_found']:
        print(f"  Issues addressed: {len(cleaning_report['issues_found'])}")
        for issue in cleaning_report['issues_found'][:5]:  # Show first 5
            print(f"    • {issue}")
        if len(cleaning_report['issues_found']) > 5:
            print(f"    ... and {len(cleaning_report['issues_found']) - 5} more")
    
    return cleaned_df, cleaning_report

print("Comprehensive data cleaning functions created!")
print("Features: Advanced test detection, statistical validation, quality scoring")